In [9]:
import pandas as pd

orders = pd.read_csv(r"C:\Users\DEEP\OneDrive\Documents\Hackathon\Dataset\orders.csv")
orders.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [10]:
users = pd.read_json(r"C:\Users\DEEP\OneDrive\Documents\Hackathon\Dataset\users.json")
users.head()


,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [12]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

with open(r"C:\Users\DEEP\OneDrive\Documents\Hackathon\Dataset\restaurants.sql", "r") as file:
    sql_script = file.read()

cursor.executescript(sql_script)

restaurants = pd.read_sql("SELECT * FROM restaurants", conn)
restaurants.head()


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [13]:
## Join Order and Users
orders_users = pd.merge(
    orders,
    users,
    on="user_id",
    how="left"
)



In [ ]:
## Join the above result with Restaurants
final_df = pd.merge(
    orders_users,
    restaurants,
    on="restaurant_id",
    how="left"
)



In [15]:
## Final Dataset Export
final_df.to_csv("final_food_delivery_dataset.csv", index=False)


In [16]:
## Order Trends Over Time
final_df['order_date'] = pd.to_datetime(final_df['order_date'])
final_df.groupby(final_df['order_date'].dt.month)['order_id'].count()


C:\Users\DEEP\AppData\Local\Temp\ipykernel_22688\501030479.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  final_df['order_date'] = pd.to_datetime(final_df['order_date'])


order_date
1     831
2     785
3     903
4     812
5     844
6     784
7     859
8     851
9     812
10    863
11    807
12    849
Name: order_id, dtype: int64

In [17]:
final_df.groupby("user_id")["order_id"].count().sort_values(ascending=False)


user_id
2973    13
1515    12
496     11
874     11
1337    11
        ..
2961     1
2957     1
2951     1
1276     1
1272     1
Name: order_id, Length: 2883, dtype: int64

In [21]:
## city wise and cuisine wise Performance
final_df.groupby("city")["total_amount"].sum()      # User city
final_df.groupby("cuisine")["total_amount"].sum()     # Cuisine revenue


cuisine
Chinese    1930504.65
Indian     1971412.58
Italian    2024203.80
Mexican    2085503.09
Name: total_amount, dtype: float64

In [22]:
##Membership impact (Gold vs Regular)
final_df.groupby("membership")["total_amount"].mean()


membership
Gold       797.145556
Regular    805.158434
Name: total_amount, dtype: float64

In [24]:
##Revenue Distribution & Seasonality
final_df.groupby(final_df['order_date'].dt.day_name())['total_amount'].sum()

order_date
Friday       1174418.88
Monday       1137568.63
Saturday     1156335.84
Sunday       1189055.72
Thursday     1145695.14
Tuesday      1082081.63
Wednesday    1126468.28
Name: total_amount, dtype: float64

In [ ]:
##City with highest revenue from Gold members
gold_city = (
    final_df[final_df["membership"] == "Gold"]
    .groupby("city")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)
print(gold_city)


city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64


In [ ]:
##Cuisine with highest average order value
final_df.groupby("cuisine")["total_amount"].mean().sort_values(ascending=False)


cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

In [28]:
##Distinct users spending > ₹1000 total
high_spenders = (
    final_df.groupby("user_id")["total_amount"]
    .sum()
)
count = (high_spenders > 1000).sum()
print(count)

2544


In [29]:
##Rating range with highest revenue
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ["3.0–3.5","3.6–4.0","4.1–4.5","4.6–5.0"]

final_df["rating_range"] = pd.cut(final_df["rating"], bins=bins, labels=labels)

final_df.groupby("rating_range")["total_amount"].sum().sort_values(ascending=False)


C:\Users\DEEP\AppData\Local\Temp\ipykernel_22688\2412512069.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  final_df.groupby("rating_range")["total_amount"].sum().sort_values(ascending=False)


rating_range
4.6–5.0    2197030.75
4.1–4.5    1960326.26
3.0–3.5    1881754.57
3.6–4.0    1717494.41
Name: total_amount, dtype: float64

In [31]:
##Gold members — city with highest average order value
final_df[final_df["membership"]=="Gold"] \
    .groupby("city")["total_amount"] \
    .mean() \
    .sort_values(ascending=False)

city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64

In [32]:
##Cuisine with lowest restaurant count but high revenue
restaurants_per_cuisine = final_df.groupby("cuisine")["restaurant_id"].nunique()
revenue_per_cuisine = final_df.groupby("cuisine")["total_amount"].sum()

pd.concat([restaurants_per_cuisine, revenue_per_cuisine], axis=1)

,restaurant_id,total_amount
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [33]:
##% of orders by Gold members
pct = (final_df["membership"]=="Gold").mean() * 100
round(pct)

50

In [40]:
##Restaurant with high AOV but <20 orders
rest_stats = final_df.groupby("restaurant_id").agg(
    restaurant_name=("restaurant_name_y", "first"),
    avg_order_value=("total_amount","mean"),
    total_orders=("order_id","count")
)

rest_stats[rest_stats["total_orders"] < 20] \
    .sort_values("avg_order_value", ascending=False)


,restaurant_name,avg_order_value,total_orders
restaurant_id,,,
294,Restaurant_294,1040.222308,13
262,Restaurant_262,1029.473333,18
77,Restaurant_77,1029.180833,12
193,Restaurant_193,1026.306667,15
7,Restaurant_7,1002.140625,16
...,...,...,...
184,Restaurant_184,621.828947,19
498,Restaurant_498,596.815556,18
192,Restaurant_192,589.972857,14


In [42]:
##Highest revenue combination
final_df.groupby(["membership","cuisine"])["total_amount"].sum().sort_values(ascending=False)

membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [43]:
##Highest revenue quarter
final_df["order_date"] = pd.to_datetime(final_df["order_date"])
final_df["quarter"] = final_df["order_date"].dt.quarter

final_df.groupby("quarter")["total_amount"].sum().sort_values(ascending=False)

quarter
3    2037385.10
4    2018263.66
1    2010626.64
2    1945348.72
Name: total_amount, dtype: float64

In [44]:
##Total orders by Gold members
gold_orders = final_df[final_df["membership"] == "Gold"].shape[0]
print(gold_orders)

4987


In [46]:
##Total revenue from Hyderabad
round(
    final_df[final_df["city"] == "Hyderabad"]["total_amount"].sum()
)

1889367

In [47]:
##Distinct users who placed at least one order
final_df["user_id"].nunique()

2883

In [48]:
##Average order value for Gold members
round(
    final_df[final_df["membership"] == "Gold"]["total_amount"].mean(),
    2
)

np.float64(797.15)

In [49]:
##Orders for restaurants with rating ≥ 4.5
final_df[final_df["rating"] >= 4.5].shape[0]

3374

In [52]:
##Orders in the top revenue city among Gold members


top_city = (
    final_df[final_df["membership"]=="Gold"]
    .groupby("city")["total_amount"]
    .sum()
    .idxmax()
)


final_df[
    (final_df["membership"]=="Gold") &
    (final_df["city"]==top_city)
].shape[0]

1337